# Finetune OCR biển số VN — DatVision GT

**Cách dùng:** Runtime > Change runtime type > **GPU (T4)**. Rồi **Runtime > Run all**.
Ở *Bước 2* sẽ hiện nút chọn file — chọn **`vn-ocr-finetune.zip`** (dataset).
Cuối cùng notebook tự tải về `model.onnx` + config — gửi 2 file đó lại để deploy.


In [ ]:
!pip install -q "fast-plate-ocr[train,onnx]"
!pip install -q "tensorflow[and-cuda]"


In [ ]:
import os
os.environ['KERAS_BACKEND']='tensorflow'
os.environ['TF_CPP_MIN_LOG_LEVEL']='3'


## 1) Tải model gốc cct-xs-v2 (weights + config) để finetune


In [ ]:
!wget -q https://github.com/ankandrew/fast-plate-ocr/releases/download/arg-plates/cct_xs_v2_global.keras
!wget -q https://github.com/ankandrew/fast-plate-ocr/releases/download/arg-plates/cct_xs_v2_global_plate_config.yaml
!wget -q https://github.com/ankandrew/fast-plate-ocr/releases/download/arg-plates/cct_xs_v2_global_model_config.yaml
!ls -la *.keras *.yaml


## 2) Upload dataset — chọn `vn-ocr-finetune.zip`


In [ ]:
from google.colab import files
files.upload()
!unzip -q -o vn-ocr-finetune.zip -d /content
!ls /content/vn-ocr-finetune


In [ ]:
# Chuẩn hoá image_path về dạng 'images/<tên>' (loader tự ghép thư mục CSV vào)
import pandas as pd, os
DS='/content/vn-ocr-finetune'
for name in ['train.csv','val.csv']:
    df=pd.read_csv(f'{DS}/{name}')
    df['image_path']=df['image_path'].apply(lambda p: 'images/'+os.path.basename(str(p)))
    df.to_csv(f'{DS}/{name}', index=False)
print('train:', len(pd.read_csv(f'{DS}/train.csv')), '| val:', len(pd.read_csv(f'{DS}/val.csv')))
pd.read_csv(f'{DS}/train.csv').head()


## 3) Augmentation nhắm đúng lỗi: tem che (CoarseDropout), chói (brightness/shadow), mờ (blur)


In [ ]:
import albumentations as A
# Augmentation mô phỏng đúng lỗi thực tế: che 1 phần (tem), chói 1 phần, mờ, nghiêng
transform = A.Compose([
    A.Affine(scale=(0.88,1.12), rotate=(-8,8), shear=(-5,5), p=0.6),          # góc/khoảng cách
    A.RandomBrightnessContrast(0.4, 0.35, p=0.7),                              # sáng/tối
    A.RandomShadow(p=0.3),                                                      # bóng đổ 1 phần
    A.RandomSunFlare(p=0.25),                                                   # chói sáng 1 vùng
    A.CoarseDropout(p=0.5),                                                      # TEM che 1 phần ký tự
    A.OneOf([A.MotionBlur(blur_limit=5), A.GaussianBlur(blur_limit=5)], p=0.45),# mờ chuyển động
    A.GaussNoise(p=0.25),                                                        # nhiễu
])
A.save(transform, 'custom_augmentation.yaml', data_format='yaml')
print('saved custom_augmentation.yaml')


## 4) Finetune (từ weights v2, learning-rate nhỏ để không phá kiến thức cũ)


In [ ]:
!KERAS_BACKEND=tensorflow fast-plate-ocr train \
  --model-config-file ./cct_xs_v2_global_model_config.yaml \
  --plate-config-file ./cct_xs_v2_global_plate_config.yaml \
  --annotations /content/vn-ocr-finetune/train.csv \
  --val-annotations /content/vn-ocr-finetune/val.csv \
  --weights-path ./cct_xs_v2_global.keras \
  --augmentation-path ./custom_augmentation.yaml \
  --epochs 20 --batch-size 32 --lr 5e-5 --weight-decay 5e-4 \
  --validate-dataset warn --output-dir /content/trained_models


## 5) Export ONNX + tải về (gửi lại 2 file cho mình)


In [ ]:
import glob, os
os.makedirs('/content/export', exist_ok=True)
cands = sorted(glob.glob('/content/trained_models/**/best.keras', recursive=True)) or sorted(glob.glob('/content/trained_models/**/*.keras', recursive=True))
assert cands, 'Khong tim thay model da train - kiem tra cell train phia tren.'
best = cands[-1]
print('best model:', best)
!fast-plate-ocr export --format onnx --simplify --plate-config-file ./cct_xs_v2_global_plate_config.yaml --model "$best" --save-dir /content/export
!ls -la /content/export


In [ ]:
from google.colab import files
import glob
onnx = sorted(glob.glob('/content/export/**/*.onnx', recursive=True))
assert onnx, 'Chua co ONNX - xem cell export phia tren co bao loi khong.'
print('ONNX:', onnx[-1])
files.download(onnx[-1])
try:
    files.download('/content/cct_xs_v2_global_plate_config.yaml')
except Exception:
    pass  # chi can best.onnx la du
print('Xong! Tai best.onnx ve gui lai cho minh.')
